# Business Entity Resolution — Learning Project

Goal: learn ML/DS fundamentals end-to-end by solving a real Entity Resolution problem.

**Pipeline overview:**
1. Load & Explore
2. Understand the "messiness" (name/address noise)
3. Blocking / Candidate Generation
4. Feature Engineering (string similarity)
5. Model Training (binary classifier: match / not match)
6. Validation Split + F0.5 Evaluation
7. Generate `matching_results.tsv` and `candidate_pairs.tsv` for test set

We'll fill in each section together, one at a time. Run cells top to bottom as we go;
don't skip ahead to sections marked TODO until we get there.


## Step 1: Load & Explore

Goal: get a feel for the data before writing a single line of ML code.

In [1]:
import pandas as pd

# sep="\t" is critical here -- these are TSV files, not CSV.
# If you forget this, pandas will silently load the whole line as ONE column
# and everything downstream will look broken in confusing ways.
s1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")


### 1.1 Basic shape
`.shape` returns `(num_rows, num_columns)` — tells you scale at a glance.

In [2]:
print("Source 1 shape:", s1.shape)
print("Source 2 shape:", s2.shape)
print("Source 3 shape:", s3.shape)
print("Ground truth shape:", gt.shape)


Source 1 shape: (1048575, 4)
Source 2 shape: (5034616, 4)
Source 3 shape: (5285603, 4)
Ground truth shape: (2206821, 2)


### 1.2 Peek at the rows
`.head()` shows the first 5 rows — sanity check that the tab-separated read worked (you should see 4 distinct columns, not 1 giant one).

In [3]:
s1.head()

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [4]:
gt.head()

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


### 1.3 Data types and missing values
`.info()` shows dtype + non-null count per column. Watch for nulls in `business_address` or `country` — missing data changes how we engineer features later.

In [5]:
s1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 4 columns):
 #   Column            Non-Null Count    Dtype
---  ------            --------------    -----
 0   entity_id         1048575 non-null  str  
 1   business_name     1048575 non-null  str  
 2   business_address  1048575 non-null  str  
 3   country           1048575 non-null  str  
dtypes: str(4)
memory usage: 32.0 MB


### 1.4 Country distribution
`.value_counts()` counts rows per category. Confirm US/India split in train, and remember: test also has France, which we must NOT hardcode against.

In [ ]:
print("Source 1 country counts:")
print(s1["country"].value_counts())
print()
print("Source 2 country counts:")
print(s2["country"].value_counts())

Source 1 country counts:
country
US       628702
India    419873
Name: count, dtype: int64

Source 2 country counts:
country
US       3016817
India    2017799
Name: count, dtype: int64


### 1.5 Ground truth shape of the problem
`matched_entity_ids` is a comma-separated string, or empty/NaN for singletons.
Let's count: how many Source-1 entities have ZERO matches (singletons) vs. some matches?

In [7]:
gt["is_singleton"] = gt["matched_entity_ids"].isna() | (gt["matched_entity_ids"] == "")
print("Singleton count:", gt["is_singleton"].sum())
print("Non-singleton count:", (~gt["is_singleton"]).sum())
print("Singleton fraction:", gt["is_singleton"].mean())


Singleton count: 123247
Non-singleton count: 2083574
Singleton fraction: 0.05584820880352326


### 1.6 How many matches per non-singleton entity?
We split the comma-separated string into a list and count its length using `.apply()` — runs a function on every row.

In [ ]:
def count_matches(x):
    if pd.isna(x) or x == "":
        return 0
    return len(x.split(","))

gt["num_matches"] = gt["matched_entity_ids"].apply(count_matches)
print(gt["num_matches"].value_counts().sort_index())
print()
print("Max matches for a single entity:", gt["num_matches"].max())


num_matches
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

Max matches for a single entity: 11


: 

**Pause here.** Run everything above and look at the printed output before moving on — 
we'll discuss what these numbers mean for our strategy before touching Step 2.

## Step 2: Understand the Noise (TODO — we'll build this together next)

Goal: look at real examples of name/address noise (abbreviations, typos, reordering, etc.)
so we know what our similarity features need to handle.


## Step 3: Blocking / Candidate Generation (TODO)

Goal: for each Source 1 entity, cheaply narrow down a small set of plausible Source 2 / Source 3
candidates. This sets our recall ceiling — nothing later can recover a match blocking misses.


## Step 4: Feature Engineering (TODO)

Goal: turn each (Source1, candidate) pair into numeric similarity features
(Jaccard, Levenshtein, TF-IDF cosine, etc. on name and address).


## Step 5: Model Training (TODO)

Goal: train a binary classifier (match / not match) on labeled pairs built from
`train_ground_truth.tsv` + our candidate pairs.


## Step 6: Validation Split + F0.5 Evaluation (TODO)

Goal: since test has no ground truth, we carve out our own validation slice of train
to self-score before submitting. Implement the F0.5 formula from the problem statement.


## Step 7: Generate Submission Files (TODO)

Goal: produce `output/matching_results.tsv` and `output/candidate_pairs.tsv` for the test set,
then validate with `utils/validate_submission.py`.
